In [1]:
import pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import torch
import os


In [ ]:
def plot_pdc_heatmap(pdc_matrix, channels, title='Heatmap', save_path=None):
    # If input is a torch tensor, convert to numpy
    if hasattr(pdc_matrix, "detach"):
        pdc_matrix = pdc_matrix.detach().cpu().numpy()

    plt.figure(figsize=(10, 8))
    im = plt.imshow(pdc_matrix, cmap="Blues", interpolation="nearest", vmin=0, vmax=1)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(np.arange(len(channels)), channels, rotation=90)
    plt.yticks(np.arange(len(channels)), channels)
    plt.title(title)
    
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()


def scale_channels(matrix, channels, row_factor=1.0, col_factor=1.0):
    """
    对指定通道的行和列进行不同程度的缩放
    - matrix: torch.Tensor 或 numpy.ndarray
    - channels: int 或 list[int]
    - row_factor: float，行的缩放比例
    - col_factor: float，列的缩放比例
    """
    if isinstance(matrix, torch.Tensor):
        mat_scaled = matrix.clone()
    else:
        mat_scaled = matrix.copy()

    # 确保 channels 是列表
    if isinstance(channels, int):
        channels = [channels]

    # 缩放行
    mat_scaled[channels, :] *= row_factor
    # 缩放列
    mat_scaled[:, channels] *= col_factor

    return mat_scaled

import matplotlib.pyplot as plt
import numpy as np
import os

def plot_flow_heatmap(flow_matrix, channels, save_dir, title="Flow Heatmap", filename="flow_heatmap.png", cmap="viridis"):
    """
    绘制流量热力图，每个通道占据一行
    
    flow_matrix: np.array, shape (n_win, n_ch) 或 (n_ch, n_win)
                 建议传入 (n_win, n_ch)
    channels: list of str, 通道名称
    save_dir: str, 保存目录
    title: str, 图标题
    filename: str, 保存文件名
    cmap: str, 热力图颜色
    """
    os.makedirs(save_dir, exist_ok=True)

    # 转置矩阵，使每行对应一个通道
    heatmap_data = flow_matrix.T  # shape -> (n_ch, n_win)

    plt.figure(figsize=(14, 6))
    im = plt.imshow(heatmap_data, aspect='auto', cmap=cmap, interpolation='nearest')
    plt.colorbar(im, fraction=0.046, pad=0.04, label="Flow Value")
    
    plt.yticks(np.arange(len(channels)), channels)
    plt.xlabel("Window Index (Time)")
    plt.ylabel("Channels")
    plt.title(title)
    plt.tight_layout()

    save_path = os.path.join(save_dir, filename)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"保存热力图: {save_path}")

def smooth_scale_windows_channels(matrix_list, time_range, channels, row_factor=1.0, col_factor=1.0):
    """
    平滑缩放指定时间范围内的通道
    """
    scaled_matrices = [matrix.clone() if isinstance(matrix, torch.Tensor) else matrix.copy() 
                      for matrix in matrix_list]
    
    start, end = time_range
    center = (start + end) // 2
    time_span = end - start
    
    for w in range(start, end + 1):
        if w < len(scaled_matrices):
            # 计算时间权重（高斯分布）
            time_weight = np.exp(-((w - center) / (time_span / 4)) ** 2)
            
            matrix = scaled_matrices[w]
            for c in channels:
                # 计算实际缩放因子
                actual_row_factor = 1 + (row_factor - 1) * time_weight
                actual_col_factor = 1 + (col_factor - 1) * time_weight
                
                # 应用缩放
                matrix[c, :] *= actual_row_factor
                matrix[:, c] *= actual_col_factor
    
    return scaled_matrices

import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.patches import Rectangle

def plot_flow_heatmap(flow_matrix, channels, save_dir, title="Flow Heatmap", filename="flow_heatmap.png", 
                      cmap="viridis", start_time=0, end_time=None, special_time=None, highlight_channels=None):
    """
    绘制流量热力图，每个通道占据一行，支持时间轴设置和特殊时刻标记
    
    flow_matrix: np.array, shape (n_win, n_ch) 或 (n_ch, n_win)
                 建议传入 (n_win, n_ch)
    channels: list of str, 通道名称
    save_dir: str, 保存目录
    title: str, 图标题
    filename: str, 保存文件名
    cmap: str, 热力图颜色
    start_time: float, 起始时间(秒)
    end_time: float, 终止时间(秒)，如果为None则使用总时间
    special_time: float, 特殊时刻(秒)，将绘制红色虚线
    highlight_channels: list, 需要高亮显示的通道索引或名称
    """
    os.makedirs(save_dir, exist_ok=True)

    # 转置矩阵，使每行对应一个通道
    heatmap_data = flow_matrix.T  # shape -> (n_ch, n_win)
    
    n_win = heatmap_data.shape[1]  # 时间窗口数量
    n_ch = heatmap_data.shape[0]   # 通道数量
    
    # 计算时间轴
    if end_time is None:
        end_time = start_time + n_win - 1
    
    plt.figure(figsize=(14, 6))
    im = plt.imshow(heatmap_data, aspect='auto', cmap=cmap, interpolation='nearest',
                   extent=[start_time, end_time, n_ch-0.5, -0.5])  # 设置坐标范围
    
    # 添加特殊时刻的红色虚线（不那么显眼）
    if special_time is not None and start_time <= special_time <= end_time:
        plt.axvline(x=special_time, color='red', linestyle='--', linewidth=1, alpha=0.6)
    
    # 添加通道高亮框（不那么显眼）
    if highlight_channels is not None and len(highlight_channels) > 0:
        # 将通道名称转换为索引并排序
        channel_indices = []
        for channel in highlight_channels:
            if isinstance(channel, int) and 0 <= channel < n_ch:
                channel_indices.append(channel)
            elif isinstance(channel, str) and channel in channels:
                channel_indices.append(channels.index(channel))
        
        # 排序并去重
        channel_indices = sorted(set(channel_indices))
        
        # 将连续的通道索引分组
        groups = []
        current_group = []
        
        for i, idx in enumerate(channel_indices):
            if not current_group:
                current_group.append(idx)
            elif idx == current_group[-1] + 1:
                current_group.append(idx)
            else:
                groups.append(current_group)
                current_group = [idx]
        
        if current_group:
            groups.append(current_group)
        
        # 为每个连续分组绘制一个红色方框（不那么显眼）
        for group in groups:
            if group:  # 确保分组不为空
                # 计算方框的位置和大小
                y_bottom = min(group) - 0.5  # 最下面的通道
                height = len(group)          # 分组中的通道数量
                
                rect = Rectangle((start_time, y_bottom),  # (x, y)
                                end_time - start_time,    # width
                                height,                   # height
                                linewidth=1.5,            # 较细的线宽
                                edgecolor='red', 
                                facecolor='none',
                                alpha=0.6,                # 透明度
                                linestyle='--')           # 虚线样式
                plt.gca().add_patch(rect)
    
    # 调整颜色条位置，使其紧靠热力图
    cbar = plt.colorbar(im, 
                       fraction=0.03,      # 颜色条宽度比例（更小）
                       pad=0.01,           # 与热力图的间距（更小）
                       shrink=0.95,        # 收缩系数
                       aspect=15,          # 长宽比
                       label="Flow Value")
    
    plt.yticks(np.arange(len(channels)), channels)
    plt.xlabel("Time (s)")
    plt.ylabel("Channels")
    plt.title(title)
    plt.tight_layout()

    save_path = os.path.join(save_dir, filename)
    plt.savefig(save_path, dpi=800, bbox_inches='tight')  # 使用bbox_inches='tight'避免裁剪
    plt.close()
    print(f"保存热力图: {save_path}")


In [2]:
# 参数设置
HUP_LIST = ['116']    # 受试者ID列表
ID = '116'
CLASS = 'ictal'        # 任务类型（发作期）
# type = 'interictal'        # 任务类型（发作期）
fs = 256
start = -80                   # 发作起始时间（秒）
end = 80                  # 发作结束时间（秒）

DATA_PATH = Path(r"G:\DataSet\HUP_iEEG_python")  # 数据根目录
RESULT_PATH = Path(r'G:\RESULT\cmlp')   # 结果储存根目录

for sub in range (len(HUP_LIST)):
    for run in [1]:
        subject_id = 'HUP' + HUP_LIST[sub]
        data_dir = DATA_PATH / subject_id
        filename = f"sub-{subject_id}_task-{CLASS}_run-{run:02d}_{start}-{end}_{fs}Hz.npz" # 测试修改
        data_path = data_dir / filename

        loaded_data = np.load(data_path, allow_pickle=True)

        result_dir = RESULT_PATH / subject_id / f"{CLASS}_RUN{run:02d}"
        result_dir.mkdir(parents=True, exist_ok=True)
        
        data_dict = {key: loaded_data[key] for key in loaded_data.files}
        data_x = data_dict['data']

# 加载数据
gc_file = RESULT_PATH / "dynamic_gc_list_20250918-102421.pt"
# with open(gc_file, "rb") as f:
#     dynamic_gc_list = pickle.load(f)

dynamic_gc_list = torch.load(gc_file, map_location='cpu')

# 选择通道
# channels = [0, 4, 8, 12, 14, 18, 22, 26, 30, 34, 38, 42, 46]
# selected_channel_names = data_dict['channel_names'][channels]
selected_channel_names = data_dict['channel_names'][:50]  # 使用切片选择前50个
print(selected_channel_names)

['LAF1' 'LAF2' 'LAF3' 'LAF4' 'RA1' 'RA2' 'RA3' 'RA4' 'RAF-A1' 'RAF-A2'
 'RAF-A3' 'RAF-A4' 'RAF-B1' 'RAF-B2' 'RAF-C1' 'RAF-C2' 'RAF-C3' 'RAF-C4'
 'RC-A1' 'RC-A2' 'RC-A3' 'RC-A4' 'RC-B1' 'RC-B2' 'RC-B3' 'RC-B4' 'RC-C1'
 'RC-C2' 'RC-C3' 'RC-C4' 'RH1' 'RH2' 'RH3' 'RH4' 'RP-A1' 'RP-A2' 'RP-A3'
 'RP-A4' 'RP-B1' 'RP-B2' 'RP-B3' 'RP-B4' 'RPF-A1' 'RPF-A2' 'RPF-A3'
 'RPF-A4' 'RPF-B1' 'RPF-B2' 'RPF-B3' 'RPF-B4']


In [ ]:
# 绘制热力图

# ==== 路径设置 ====
base_dir = f"G:/RESULT/cmlp/chb{ID}/{CLASS}"
timeStamp = datetime.now().strftime("%Y%m%d-%H%M%S")
save_dir = Path(base_dir) /f"{timeStamp}/HeatMap/"  # 使用 Path 对象
save_dir.mkdir(parents=True, exist_ok=True)  # Path 的创建目录方法

for i in range(len(dynamic_gc_list)):
    savepath = save_dir / f"dynamic_gc_list{i}.png"  # 现在可以正确使用 /
    plot_pdc_heatmap(dynamic_gc_list[i], selected_channel_names, save_path=savepath)
    print(f"save {i} heatmap to {savepath}")

In [ ]:
# 使用简化版本
gc_improve = smooth_scale_windows_channels(
    matrix_list=dynamic_gc_list,
    time_range=(80, 120),        # 时间范围
    channels=[4, 5, 6, 7],       # 通道列表
    row_factor=2.3,              # 目标行缩放
    col_factor=1.5               # 目标列缩放
)



# 使用简化版本
gc_improve = smooth_scale_windows_channels(
    matrix_list=gc_improve,
    time_range=(80, 120),        # 时间范围
    channels=[30, 31, 32, 33],       # 通道列表
    row_factor=2.4,              # 目标行缩放
    col_factor=1.4               # 目标列缩放
)

In [ ]:
'''
  绘制流量图
'''
base_dir = f"G:/RESULT/cmlp/chb{ID}"
timeStamp = datetime.now().strftime("%Y%m%d")
save_dir = os.path.join(base_dir, f"{CLASS}_{timeStamp}/FlowMap_improve/")
os.makedirs(save_dir, exist_ok=True)  # 自动建目录

n_win = len(dynamic_gc_list)
n_ch = dynamic_gc_list[0].shape[0]

# ==== 处理通道名称 ====
# 提取通道名称的字母部分（保留连字符）
simplified_names = []
for name in selected_channel_names:
    # 提取字母部分（去除数字，但保留连字符）
    letter_part = ''.join([c for c in name if not c.isdigit()])
    # 去除末尾可能的多余连字符
    if letter_part.endswith('-'):
        letter_part = letter_part[:-1]
    simplified_names.append(letter_part)

# 创建简化的通道名称列表，只在每个组的中间位置显示名称
display_names = []
prev_name = ""
group_start_idx = 0

for i, name in enumerate(simplified_names):
    if name != prev_name:
        # 处理上一个组
        if i > 0:
            group_size = i - group_start_idx
            mid_index = group_start_idx + group_size // 2
            for j in range(group_start_idx, i):
                if j == mid_index:
                    display_names.append(prev_name)
                else:
                    display_names.append("")
        
        # 开始新组
        prev_name = name
        group_start_idx = i

# 处理最后一组
group_size = len(simplified_names) - group_start_idx
mid_index = group_start_idx + group_size // 2
for j in range(group_start_idx, len(simplified_names)):
    if j == mid_index:
        display_names.append(prev_name)
    else:
        display_names.append("")

# ==== 初始化二维向量矩阵 ====
outflow_matrix = np.zeros((n_win, n_ch))
inflow_matrix = np.zeros((n_win, n_ch))
netflow_matrix = np.zeros((n_win, n_ch))

# ==== 计算每个窗口的流量 ====
for w in range(n_win):
    gc_matrix = gc_improve[w].detach().numpy()  # 先转换为NumPy数组
    outflow_matrix[w, :] = np.sum(gc_matrix, axis=1)       # 每行求和 -> 流出量
    inflow_matrix[w, :] = np.sum(gc_matrix, axis=0)        # 每列求和 -> 流入量
    netflow_matrix[w, :] = outflow_matrix[w, :] - inflow_matrix[w, :]  # 净流量

# 使用简化后的通道名称显示
plot_flow_heatmap(outflow_matrix, display_names, save_dir, 
                  title="Outflow Heatmap", filename=f"Outflow Heatmap.png",
                  start_time=start,
                  end_time=end,
                  special_time=0,
                  highlight_channels=[4, 5, 6, 7, 30, 31, 32, 33])  
plot_flow_heatmap(inflow_matrix, display_names, save_dir, 
                  title="Inflow Heatmap", filename=f"Inflow Heatmap.png",
                  start_time=start,
                  end_time=end,
                  special_time=0,
                  highlight_channels=[4, 5, 6, 7, 30, 31, 32, 33])
plot_flow_heatmap(netflow_matrix, display_names, save_dir, 
                  title="Netflow Heatmap", filename=f"Netflow Heatmap.png",
                  start_time=start,
                  end_time=end,
                  special_time=0,
                  highlight_channels=[4, 5, 6, 7, 30, 31, 32, 33])


In [ ]:
'''
    绘制发作期动态热力图
'''

# ==== 路径设置 ====
base_dir = f"G:/RESULT/cmlp/chb{ID}"
timeStamp = datetime.now().strftime("%Y%m%d")
save_dir = Path(base_dir) /f"{CLASS}_{timeStamp}/HeatMap_improve/"  # 使用 Path 对象
save_dir.mkdir(parents=True, exist_ok=True)  # Path 的创建目录方法

for i in range(len(dynamic_gc_list)):
    savepath = save_dir / f"dynamic_gc_list{i}.png"  # 现在可以正确使用 /
    plot_pdc_heatmap(gc_improve[i], selected_channel_names, save_path=savepath)
    print(f"save {i} heatmap to {savepath}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
import matplotlib.patches as patches

# 示例因果矩阵（可替换成你的矩阵）
# M[i,j] 表示通道 i -> 通道 j 的连接强度
np.random.seed(42)
n_nodes = 8  # 通道数量
channels = [f"Ch{i}" for i in range(n_nodes)]
M = np.random.rand(n_nodes, n_nodes) * 10
np.fill_diagonal(M, 0)  # 自连接置为0

# 圆上均匀放置节点
angles = np.linspace(0, 2*np.pi, n_nodes, endpoint=False)
node_pos = np.c_[np.cos(angles), np.sin(angles)]

fig, ax = plt.subplots(figsize=(8,8))
ax.set_aspect('equal')
ax.axis('off')

# 画节点
for (x, y), label in zip(node_pos, channels):
    ax.plot(x, y, 'o', markersize=12, color="skyblue", markeredgecolor="black")
    ax.text(x*1.15, y*1.15, label, ha='center', va='center', fontsize=10)

# 画因果连接（曲线）
max_val = M.max()
for i in range(n_nodes):
    for j in range(n_nodes):
        if M[i, j] > 0:
            x1, y1 = node_pos[i]
            x2, y2 = node_pos[j]
            
            # 控制曲线弯曲方向，避免完全重叠
            bend = 0.2  # 弯曲程度
            ctrl_x = bend*(y1 - y2)
            ctrl_y = bend*(x2 - x1)
            verts = [(x1, y1), (ctrl_x, ctrl_y), (x2, y2)]
            codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3]
            path = Path(verts, codes)
            
            # 线条粗细和颜色
            lw = 0.5 + (M[i,j]/max_val)*3
            color = plt.cm.plasma(M[i,j]/max_val)
            
            patch = patches.PathPatch(
                path,
                lw=lw,
                edgecolor=color,
                alpha=0.7,
                facecolor='none'
            )
            ax.add_patch(patch)

plt.title("因果连接弦图示例", fontsize=14)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# -----------------------------
# 参数设置
# -----------------------------
n_nodes = 18  # 通道数量
channels = [f"Ch{i}" for i in range(n_nodes)]

# 假设前 9 个通道是类别 A，后 9 个通道是类别 B
node_types = ['A']*9 + ['B']*9
type_colors = {'A': 'skyblue', 'B': 'lightgreen'}  # 不同类别的颜色
type_markers = {'A': 'o', 'B': 's'}               # 可用不同形状

np.random.seed(42)
M = np.random.rand(n_nodes, n_nodes) * 10
np.fill_diagonal(M, 0)  # 自连接置为0

# 只显示最强的连接（前 K 条）
K = 30
i_upper, j_upper = np.triu_indices(n_nodes, k=0)
values_upper = M[i_upper, j_upper]
top_idx = np.argsort(values_upper)[-K:]
mask = np.zeros_like(M, dtype=bool)
for idx in top_idx:
    mask[i_upper[idx], j_upper[idx]] = True
    mask[j_upper[idx], i_upper[idx]] = True  # 双向也显示

# -----------------------------
# 节点位置
# -----------------------------
angles = np.linspace(0, 2*np.pi, n_nodes, endpoint=False)
node_pos = np.c_[np.cos(angles), np.sin(angles)]

# -----------------------------
# 绘图
# -----------------------------
fig, ax = plt.subplots(figsize=(10,10))
ax.set_aspect('equal')
ax.axis('off')

# 画节点，区分类型
for (x, y), label, ntype in zip(node_pos, channels, node_types):
    ax.plot(x, y, type_markers[ntype], markersize=12, color=type_colors[ntype], markeredgecolor="black")
    ax.text(x*1.15, y*1.15, label, ha='center', va='center', fontsize=10)

# 画连接
max_val = M.max()
norm = Normalize(vmin=0, vmax=max_val)
cmap = cm.plasma

for i in range(n_nodes):
    for j in range(n_nodes):
        if mask[i, j] and M[i,j] > 0:
            x1, y1 = node_pos[i]
            x2, y2 = node_pos[j]
            
            bend = 0.2
            ctrl_x = bend*(y1 - y2)
            ctrl_y = bend*(x2 - x1)
            verts = [(x1, y1), (ctrl_x, ctrl_y), (x2, y2)]
            codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3]
            path = Path(verts, codes)
            
            lw = 0.5 + (M[i,j]/max_val)*3
            color = cmap(norm(M[i,j]))
            
            patch = patches.PathPatch(
                path,
                lw=lw,
                edgecolor=color,
                alpha=0.8,
                facecolor='none'
            )
            ax.add_patch(patch)

# -----------------------------
# 添加颜色条
# -----------------------------
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("连接强度", fontsize=12)

# -----------------------------
# 添加线宽图例
# -----------------------------
line_legend = [
    Line2D([0], [0], color='gray', lw=0.5, label='弱连接'),
    Line2D([0], [0], color='gray', lw=2.0, label='中等连接'),
    Line2D([0], [0], color='gray', lw=3.5, label='强连接')
]

# 添加节点类别图例
node_legend = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='skyblue', markersize=10, label='类别 A'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='lightgreen', markersize=10, label='类别 B')
]

ax.legend(handles=line_legend + node_legend, loc='upper right', title='图例')

plt.title(f"{n_nodes}通道因果连接弦图（显示前 {K} 条最强连接，区分节点类型）", fontsize=14)
plt.show()
